# CNN + LSTM Gesture Classifier

## Hybrid Architecture Overview

This notebook implements a **CNN+LSTM hybrid** model for gesture recognition on data glove time-series. The architecture follows a two-stage paradigm:

1. **1D CNN frontend** — stacked `Conv1D` layers with `BatchNormalization`, `MaxPooling1D`, and `Dropout` scan the time axis with a sliding kernel. Each filter learns to detect a local temporal pattern (e.g., a sharp acceleration spike, a flex plateau). The output is a condensed feature sequence of shape `(T', F_cnn)`.

2. **LSTM backend** — the CNN's output sequence is fed directly into stacked LSTM layers. Because the CNN has already extracted local features and reduced temporal resolution, the LSTM only needs to model the *long-range ordering* of these high-level features — which is precisely what recurrent networks excel at.

3. **Dense head** — a fully-connected layer followed by softmax classification.

### Why CNN before LSTM?

A pure LSTM must learn both local and global temporal patterns simultaneously, which is hard and slow. A pure CNN lacks the ability to retain information across long sequences. The hybrid exploits both strengths:

| Component | Learns | Limitation alone |
|-----------|--------|------------------|
| CNN only  | Local waveform patterns | Misses long-range ordering |
| LSTM only | Long-range dependencies | Struggles with local feature extraction |
| **CNN+LSTM** | **Local patterns → global sequence ordering** | Best of both |

### Literature Context

- Studies comparing CNN-LSTM against pure LSTM on sensor time-series report CNN-LSTM reaching **93.33% test accuracy** vs. pure LSTM baselines in gesture recognition tasks, attributing the gain to the CNN's ability to pre-filter noise and compress uninformative timesteps before the recurrent stage.
- The **A-CBLN paper** (*Attention-based CNN-BiLSTM Network*) uses a CNN-LSTM block as its baseline, demonstrating that the hybrid consistently outperforms single-modality sequence models on wearable sensor data.
- This notebook implements the CNN-LSTM baseline configuration described in both works, adapted to the ThesisA data glove column schema.

---

**Column naming convention:**
```
{hand}_{segment}_{loc}_{channel}

Examples:
  left_thumb_mid_ax          right_index_prox_pitch
  left_thumb_flex_mcp        right_palm_az
  left_wrist_roll
```

Quaternion columns (`qw`, `qx`, `qy`, `qz`) are always excluded.


---
## Cell 1 — Install Dependencies

Run this cell once to install (or upgrade) the required Python packages.
All subsequent cells depend on these libraries, so execute this first.


In [ ]:
"""
Install required Python packages silently.

Why these packages?
  numpy / pandas : numerical arrays and data-frame manipulation
  scipy          : signal processing (optional temporal filtering)
  scikit-learn   : train/val/test split, metrics (accuracy, confusion matrix, F1)
  matplotlib     : plotting training curves and confusion matrices
  seaborn        : heatmap visualisation of confusion matrices
  tensorflow     : the deep learning framework — provides Keras layers, optimisers,
                   callbacks, and the GPU/CPU execution backend
"""
import subprocess, sys

pkgs = [
    'numpy', 'pandas', 'scipy',
    'scikit-learn', 'matplotlib', 'seaborn',
    'tensorflow',
]

# install quietly — '--quiet' suppresses progress bars to reduce notebook noise
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs
)
print('Dependencies ready.')


---
## Cell 2 — Imports

We import every library used by the notebook here so that all dependencies
are visible in one place. If any import fails, the error appears early and
clearly before any lengthy computation begins.

**Key Keras/TensorFlow imports explained for COMP3308 students:**
- `keras.layers.Conv1D` — the 1D convolutional layer from your lecture 10 material,
  but operating on time rather than 2D images.
- `keras.layers.LSTM` — Long Short-Term Memory cell; processes sequences step by step,
  carrying a hidden state forward (explained in detail in the model cell below).
- `keras.layers.BatchNormalization` — normalises the activations across the batch after
  each Conv1D layer. Keeps activations in a stable range, accelerating training
  (similar in spirit to input normalisation you learned, but applied mid-network).
- `EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint` — training callbacks that
  monitor validation metrics and intervene automatically to prevent overfitting.


In [ ]:
"""
Import all libraries and configure the project path.

The sys.path.insert call adds the parent directory (ThesisRepo/) to Python's
module search path. This lets us use 'from utils.data_loader import ...' even
though the notebook is in ThesisRepo/ML/.
"""
import sys, os, json

# ── Path setup: add ThesisRepo root so 'utils' package is importable ──────────
# os.path.abspath("__file__") gives the notebook directory;
# os.path.dirname(..., "..") walks one level up to ThesisRepo/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

# ── Standard scientific / visualisation libraries ─────────────────────────────
import numpy as np                          # array maths; most ML data lives in numpy arrays
import pandas as pd                         # CSV loading, data-frame manipulations
import matplotlib.pyplot as plt             # plotting loss/accuracy curves
import seaborn as sns                       # heatmaps for confusion matrices

# ── Scikit-learn: data splitting and evaluation metrics ───────────────────────
from sklearn.metrics import (
    accuracy_score,          # overall proportion of correct predictions
    classification_report,   # per-class precision, recall, F1 (from COMP3308 lectures)
    confusion_matrix,        # N×N matrix: rows=true, cols=predicted
)

# ── TensorFlow / Keras: model building, training, callbacks ───────────────────
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers          # all layer types: Conv1D, LSTM, Dense, …
from tensorflow.keras.callbacks import (
    EarlyStopping,           # halt training early if val accuracy stops improving
    ReduceLROnPlateau,       # halve the learning rate when val loss plateaus
    ModelCheckpoint,         # save the best model weights to disk automatically
)

# ── Project utilities: data loading / preprocessing / splitting ───────────────
# These functions are shared across all notebooks in the project.
from utils.data_loader import (
    build_dataset,           # main entry point: loads all CSVs → (X, y) arrays
    split_dataset,           # stratified train/val/test split
    build_column_groups,     # builds the canonical feature column list from flags
    HANDS, FINGERS, LOCS,    # sensor topology constants (see data_loader.py)
    IMU_ALL_CHANNELS,        # ['ax','ay','az','yaw','pitch','roll']
    FLEX_CHANNELS,           # ['mcp_flex','pip_flex']
    WRIST_ALL_CHANNELS,      # ['ax','ay','az','heading','pitch','roll']
)

print('Imports complete.')
print(f'TensorFlow version : {tf.__version__}')
print(f'NumPy version      : {np.__version__}')


---
## Cell 3 — Configuration

All hyperparameters and data-selection flags are centralised here.
**No other cell requires manual editing** — every downstream cell reads
from these variables, so changing one value here propagates everywhere.

### What each section controls
- **DATA**: where the gesture CSV files live and how many timesteps to use.
- **SENSOR SELECTION**: toggle entire sensor groups on/off with boolean flags.
  The feature column list is built automatically from these flags.
- **PREPROCESSING**: optional temporal filtering (data is already Butterworth LP-filtered
  at the glove firmware level; additional filtering is usually not needed).
- **NORMALIZATION**: `minmax` scales each channel to [0, 1]; `zscore` standardises to
  zero mean and unit variance. Both are applied per-sample so one trial's scale
  does not affect another's.
- **DATASET SPLIT**: 70 / 10 / 20 train / val / test is standard; the random seed
  ensures reproducibility.
- **MODEL HYPERPARAMETERS**: controls the CNN depth (filters, kernel size, pooling)
  and LSTM depth (units per layer, dropout rates).
- **TRAINING**: batch size, number of epochs, learning rate, early stopping patience.


In [ ]:
"""
Central configuration cell.

Edit ONLY this cell to change experiment settings.
All other cells read from these variables — no hardcoded values elsewhere.
"""
# =============================================================================
# CONFIGURATION  —  Edit this cell to control every aspect of the pipeline
# =============================================================================

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'   # Root folder containing the 4 gesture label subfolders:
                        #   TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/
                        #   TwoHand_L_Flat_R_Flat_filtered_butterworth_lp/
                        #   TwoHand_L_Okay_R_Okay_filtered_butterworth_lp/
                        #   TwoHand_L_Rock_R_Rock_filtered_butterworth_lp/

FS_HZ      = 31.0       # Confirmed sampling rate (~156 rows / 5 s = 31.2 Hz)
TARGET_LEN = 156        # Native timesteps per trial (5 s × ~31 Hz); set lower
                        # (e.g. 78) to halve resolution and speed up training.

# ── COLUMN / SENSOR SELECTION ─────────────────────────────────────────────────
# Toggle entire sensor groups on / off.
# All combinations are valid — the feature list is built automatically.

USE_LEFT_HAND  = True    # Include left-hand sensors
USE_RIGHT_HAND = True    # Include right-hand sensors

# Per-finger IMU locations
USE_DISTAL_IMU   = True  # 'mid'  — distal phalanx IMU  (tip segment)
USE_PROXIMAL_IMU = True  # 'prox' — proximal phalanx IMU (base segment)

# IMU channel types (applies to finger IMUs AND palm_prox AND wrist)
USE_ACCELEROMETER = True  # ax, ay, az
USE_ORIENTATION   = True  # yaw, pitch, roll  (wrist uses 'heading' instead of 'yaw')

# Flex sensors  (MCP and PIP joints per finger)
USE_FLEX_SENSORS  = True  # {hand}_{finger}_{mcp_flex / pip_flex}
                           # NOTE: palm flex is always -1 (no sensor) → auto-excluded

# Back-of-hand IMU  (palm_prox — the real palm IMU; palm_mid is always zero → auto-excluded)
USE_PALM_IMU  = True

# Wrist IMU  ({hand}_wrist_{ax/ay/az/heading/pitch/roll})
USE_WRIST_IMU = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
# Temporal filter applied to each channel before training.
# The data files are already Butterworth LP-filtered at 5 Hz by the glove firmware;
# set 'none' to use them as-is, or apply an additional filter here.
#
# Options: 'none' | 'butterworth_lp' | 'butterworth_hp' | 'butterworth_bp'
#          'moving_average' | 'savgol' | 'median'
FILTER_TYPE = 'none'

# Normalization applied after resampling
# Options: 'minmax' (→ [0,1]) | 'zscore' (zero-mean unit-var) | 'none'
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True   # True = normalize each trial independently (recommended)

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70   # 70 % train   →  280 / 400 samples
VAL_RATIO   = 0.10   # 10 % val     →   40 / 400 samples
                     # 20 % test    →   80 / 400 samples (auto)
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
# CNN frontend (extracts local temporal patterns)
CNN_FILTERS     = [64, 128]   # filters per Conv1D block; 64 then 128 channels
CNN_KERNEL_SIZE = 7           # each filter spans 7 consecutive timesteps
CNN_POOL_SIZE   = 2           # MaxPool halves temporal resolution after each block
CNN_DROPOUT     = 0.2         # spatial dropout after each CNN block

# LSTM backend (models long-range temporal dependencies)
LSTM_UNITS             = [128, 64]  # two stacked LSTM layers: 128 then 64 units
LSTM_DROPOUT           = 0.3        # output dropout in LSTM (regularisation)
LSTM_RECURRENT_DROPOUT = 0.2        # Gal & Ghahramani variational recurrent dropout

# Dense head
DENSE_UNITS = [64]   # one hidden dense layer before the softmax output

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 80    # maximum training epochs (EarlyStopping may halt sooner)
BATCH_SIZE          = 32    # samples per gradient update step
LEARNING_RATE       = 1e-3  # Adam learning rate (1e-3 is Adam's conventional default)
EARLY_STOP_PATIENCE = 15    # epochs without val improvement before stopping

# ── OUTPUT PATHS ──────────────────────────────────────────────────────────────
MODEL_NAME  = '03_cnn_lstm'                 # used as filename prefix for saved files
MODEL_DIR   = 'saved_models'               # directory for .keras model files
RESULTS_DIR = 'results'                    # directory for metrics JSON and figures

# Create output directories if they don't exist yet
import os as _os
_os.makedirs(MODEL_DIR,   exist_ok=True)
_os.makedirs(RESULTS_DIR, exist_ok=True)

print('Configuration loaded.')
print(f'  CNN_FILTERS     : {CNN_FILTERS}   (two Conv1D blocks)')
print(f'  CNN_KERNEL_SIZE : {CNN_KERNEL_SIZE}  (timesteps per filter window)')
print(f'  LSTM_UNITS      : {LSTM_UNITS}   (two stacked LSTM layers)')
print(f'  TARGET_LEN      : {TARGET_LEN}  timesteps per sample')
print(f'  BATCH_SIZE      : {BATCH_SIZE}   EPOCHS: {EPOCHS}')


---
## Cell 4 — Build Feature Columns from Config

The `build_column_groups` utility constructs the canonical column list from the sensor topology.
The config flags above select which sensor groups to include. Columns that are absent
from the actual data are silently ignored at load-time via `filter_existing_columns`.

**Why build a column list first?**
Each CSV trial has 191 columns (including metadata like timestamps). We need to tell
the loader exactly which columns are sensor readings to use as features. Rather than
hardcoding 180 column names, we use the sensor topology (hands × fingers × locations ×
channels) to generate them programmatically. If you disable a sensor group (e.g.
`USE_FLEX_SENSORS = False`), the column list shrinks automatically and every downstream
cell adjusts without changes.


In [ ]:
"""
Translate CONFIG boolean flags into a concrete list of CSV column names.

After this cell, `feature_cols` is a list of strings like:
    ['left_thumb_mid_ax', 'left_thumb_mid_ay', ..., 'right_wrist_roll']

This list is passed to build_dataset() so the loader selects only the
requested columns from each CSV file.
"""
import sys, os
# Ensure the ThesisRepo root is on the path (safe to call multiple times)
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from utils.data_loader import (
    build_column_groups, HANDS, FINGERS, LOCS,
    IMU_ALL_CHANNELS, FLEX_CHANNELS, WRIST_ALL_CHANNELS,
)

# ── Resolve hand selection flags → list of hand strings ───────────────────────
# HANDS = ['right', 'left']; we keep only the ones enabled in CONFIG.
# The 'or HANDS' fallback ensures we never produce an empty list.
_hands   = ([h for h in HANDS   if (USE_LEFT_HAND   and h=="left")
                                 or (USE_RIGHT_HAND  and h=="right")]
            or HANDS)

# All finger segments are always included; per-finger disabling is not yet supported.
_fingers = FINGERS

# ── Resolve IMU location flags → ['mid', 'prox'] subset ──────────────────────
# LOCS = ['mid', 'prox']:
#   'mid'  = distal phalanx IMU (fingertip segment)
#   'prox' = proximal phalanx IMU (knuckle segment)
_locs    = [l for l in LOCS
            if (USE_DISTAL_IMU   and l=="mid")
            or (USE_PROXIMAL_IMU and l=="prox")] or LOCS

# ── Call build_column_groups to assemble the full column name list ─────────────
# This generates strings following the naming convention:
#   {hand}_{finger}_{loc}_{channel}   e.g. left_thumb_mid_ax
#   {hand}_{finger}_{flex_type}       e.g. left_index_mcp_flex
#   {hand}_wrist_{channel}            e.g. right_wrist_heading
feature_cols = build_column_groups(
    hands               = _hands,
    fingers             = _fingers,
    locs                = _locs,
    include_flex        = USE_FLEX_SENSORS,    # MCP/PIP flex sensors
    include_palm_prox   = USE_PALM_IMU,        # back-of-hand IMU
    include_wrist       = USE_WRIST_IMU,       # wrist IMU
    include_accel       = USE_ACCELEROMETER,   # ax, ay, az channels
    include_orient      = USE_ORIENTATION,     # yaw/pitch/roll channels
)

print(f"Feature columns selected: {len(feature_cols)}")
print()

# ── Group display: show a summary of columns by sensor group ──────────────────
# Parses the first 3 underscore-separated parts as the group key
# (e.g. 'left_thumb_mid'), then lists the final channel names in each group.
groups = {}
for c in feature_cols:
    parts = c.split("_")
    grp   = "_".join(parts[:3]) if len(parts) >= 4 else "_".join(parts[:2])
    groups.setdefault(grp, []).append(c)

# Print up to 12 groups to avoid flooding the output
for grp, cols in list(groups.items())[:12]:
    print(f"  {grp:40s}  ({len(cols)} cols): {[c.split('_')[-1] for c in cols]}")
if len(groups) > 12:
    print(f"  ... and {len(groups)-12} more groups")


---
## Cell 5 — Load Dataset & Class Distribution

`build_dataset` is the central data-loading function defined in `utils/data_loader.py`.
It walks `DATA_ROOT`, loads every gesture CSV, applies the configured filter,
resamples to `TARGET_LEN` timesteps, and normalises. Missing columns are handled
gracefully — the loader takes the intersection of requested columns with those
present in each file.

**What comes out:**
- `X` — a 3D NumPy array of shape `(N, TARGET_LEN, N_FEATURES)`:
  - `N` = total number of gesture trials (400 = 4 gestures × 100 trials each)
  - `TARGET_LEN` = 156 timesteps (the time axis)
  - `N_FEATURES` = number of sensor channels selected by CONFIG
- `y` — a 1D integer array of shape `(N,)`, one class label per trial (0–3)
- `CLASS_NAMES` — list mapping integers 0–3 to gesture name strings
- `FEATURE_COLS_USED` — the column names that were actually present in the data

From the COMP3308 perspective: `X` is your feature matrix and `y` is your label vector,
just in 3D (a matrix per example) rather than 1D (a vector per example).


In [ ]:
"""
Load all gesture trial CSVs into NumPy arrays.

After this cell the key variables are:
    X               : np.float32  shape (N, TARGET_LEN, N_FEATURES)
    y               : np.int64    shape (N,)  — integer class labels 0..3
    CLASS_NAMES     : list[str]   e.g. ['Fist', 'Flat', 'Okay', 'Rock']
    N_CLASSES       : int         number of gesture classes (4)
    N_FEATURES      : int         number of sensor channels used
    FEATURE_COLS_USED: list[str]  actual column names loaded from the CSVs
"""

# build_dataset orchestrates: discover → load → filter → resample → normalise → stack
# It returns (X, y, labels, feature_cols_used); labels are the gesture name strings.
X, y, CLASS_NAMES, FEATURE_COLS_USED = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = feature_cols,   # column list built in the previous cell
    filter_type     = FILTER_TYPE,    # 'none' = use data as-is (already LP filtered)
    fs              = FS_HZ,          # sampling rate used if a filter is applied
    target_len      = TARGET_LEN,     # resample every trial to this many timesteps
    normalization   = NORMALIZATION,  # 'minmax' → values in [0, 1] per channel
    per_sample_norm = PER_SAMPLE_NORM,# normalise each trial independently
)

# ── Derive key constants from the loaded data ─────────────────────────────────
N_CLASSES  = len(CLASS_NAMES)     # number of unique gesture labels (4)
N_FEATURES = X.shape[2]           # actual number of feature columns loaded

# ── Display dataset overview ──────────────────────────────────────────────────
print(f'\nDataset overview:')
print(f'  X shape       : {X.shape}')          # (N, timesteps, features)
print(f'  y shape       : {y.shape}')          # (N,)
print(f'  Classes ({N_CLASSES})  : {CLASS_NAMES}')
print(f'  N_FEATURES    : {N_FEATURES}')
print()

# ── Class distribution bar chart ──────────────────────────────────────────────
# A balanced dataset (equal samples per class) is ideal for fair evaluation.
# Imbalanced datasets can inflate accuracy (a model predicting only the majority
# class can achieve high accuracy on the metric while being useless in practice).
import numpy as _np
counts = [int((_np.array(y) == i).sum()) for i in range(N_CLASSES)]
print('Class distribution:')
for name, cnt in zip(CLASS_NAMES, counts):
    bar = '█' * (cnt // 5)   # simple ASCII bar chart (each block = 5 samples)
    print(f'  {name:20s}: {cnt:4d}  {bar}')


---
## Cell 6 — Split Dataset

Stratified split preserving class proportions across train / validation / test sets.

**Why stratify?**
Without stratification, a random split might assign all samples of one class to the
test set by chance. Stratification guarantees each split has roughly the same class
balance as the full dataset — especially important here since we have exactly 100
samples per class.

**Train / Val / Test roles (from COMP3308):**
- **Train (70%)** — the only data the model sees during `model.fit()`.
- **Validation (10%)** — used to monitor for overfitting during training (callbacks
  watch `val_accuracy` and `val_loss`). Weights are NOT updated from this set.
- **Test (20%)** — held out entirely until final evaluation. Gives an unbiased
  estimate of real-world performance.

**One-hot encoding:**
Keras's `categorical_crossentropy` loss expects class labels as *probability vectors*
rather than integers. `to_categorical(3, 4)` → `[0, 0, 0, 1]`. This is equivalent
to what you learned in lectures as a "one-of-K" target representation.


In [ ]:
"""
Stratified train / validation / test split, then one-hot encode labels.

Inputs  : X (N, T, F),  y (N,)  — from the previous cell
Outputs :
    X_train, y_train  — 70% of data (used for gradient updates)
    X_val,   y_val    — 10% of data (used for early stopping / LR decay)
    X_test,  y_test   — 20% of data (used ONLY for final evaluation)
    y_train_oh, y_val_oh, y_test_oh  — one-hot versions for categorical_crossentropy
"""
# split_dataset uses sklearn's StratifiedShuffleSplit internally,
# which guarantees equal class proportions in every split subset.
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,   # 0.70 → 280 / 400 samples for training
    val_ratio   = VAL_RATIO,     # 0.10 →  40 / 400 samples for validation
    seed        = RANDOM_SEED,   # 42   → reproducible random state
)

# One-hot encode integer labels for categorical cross-entropy loss:
#   y_train[i] is an integer 0–3  →  y_train_oh[i] is a length-4 vector
#   e.g. class 2  →  [0, 0, 1, 0]
y_train_oh = tf.keras.utils.to_categorical(y_train, N_CLASSES)  # (280, 4)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   N_CLASSES)  # ( 40, 4)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  N_CLASSES)  # ( 80, 4)

print(f'Train : {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')
print(f'Input shape per sample: ({TARGET_LEN}, {N_FEATURES})')
# Reminder: each sample is a 2D matrix of shape (timesteps, features),
# NOT a flat vector — the network processes the time axis explicitly.


---
## Cell 7 — Model Definition

### Architecture: CNN → LSTM → Dense

The model processes input sequences of shape `(TARGET_LEN, N_FEATURES)`:

```
Input  (156, F)
  │
  ├─ Conv1D(64, k=7, padding='same') → BatchNorm → ReLU
  ├─ MaxPool1D(2)                       ← T: 156 → 78
  ├─ Dropout(0.2)
  │
  ├─ Conv1D(128, k=7, padding='same') → BatchNorm → ReLU
  ├─ MaxPool1D(2)                       ← T:  78 → 39
  ├─ Dropout(0.2)
  │
  ├─ LSTM(128, return_sequences=True)   ← receives (39, 128), outputs (39, 128)
  ├─ LSTM(64,  return_sequences=False)  ← receives (39, 128), outputs (64,)
  │
  ├─ Dense(64) → ReLU → Dropout(0.3)
  └─ Dense(N_CLASSES) → Softmax         ← outputs 4 class probabilities
```

### What is an LSTM? (New concept for COMP3308 students)

You know that a Dense layer takes a fixed-size input vector and produces an output.
An LSTM is different — it processes one element of a *sequence* at a time, and
importantly it **carries information forward** between steps.

At each timestep `t`, the LSTM receives:
1. `x_t` — the current input (the 128-dimensional CNN feature vector for that step)
2. `h_{t-1}` — its own hidden state from the *previous* timestep (same dimension as
   `lstm_units`)

It uses three learned "gates" — each a small neural network layer — to decide:
- **Forget gate**: what fraction of the previous hidden state to erase
- **Input gate**: what new information from `x_t` to add to memory
- **Output gate**: what part of the current memory to expose as `h_t`

The LSTM is trained with the same backpropagation algorithm you know from COMP3308,
just applied through time (called Backpropagation Through Time, BPTT).

### return_sequences — the key switch between LSTM layers

- `return_sequences=True` : output `h_t` at **every** timestep → shape `(batch, T, units)`.
  Use this for all LSTM layers except the last; it passes the full sequence to the
  next LSTM layer.
- `return_sequences=False`: output only the **final** `h_T` → shape `(batch, units)`.
  This collapses the entire sequence into a single summary vector, ready for the Dense head.

### CNN → LSTM handoff: why this works well

The two Conv1D+Pool blocks compress the 156-step raw sensor sequence into a 39-step
sequence of 128-dimensional feature vectors. Each of the 39 "super-steps" the LSTM sees
summarises approximately 4 original timesteps (~130 ms at 31 Hz). The LSTM therefore
deals with a much shorter sequence (39 vs 156 steps), which is faster to train and
easier to model long-range dependencies across.

### BatchNormalization in the CNN stage

After each Conv1D layer, `BatchNormalization` normalises the activations across the
batch. This keeps activations in a stable range (avoiding vanishing/exploding gradients),
allows a higher learning rate, and provides mild regularisation. Because BN adds its own
learned bias term, we set `use_bias=False` in the Conv1D layers — the BN bias is
redundant.

**Key design choices:**
- `return_sequences=True` on all LSTM layers except the last passes the full hidden-state sequence forward.
- Recurrent dropout (Gal & Ghahramani) is applied *inside* the LSTM cell for regularisation.


In [ ]:
"""
Define and instantiate the CNN+LSTM hybrid model.

Shape flow through the network (with default CONFIG values):
    Input            : (batch, 156, N_FEATURES)   — raw sensor sequence
    After Conv1+Pool : (batch,  78,  64)           — 64 feature channels, half the time
    After Conv2+Pool : (batch,  39, 128)           — 128 feature channels, quarter time
    LSTM 1 (seq=True): (batch,  39, 128)           — hidden state at every step
    LSTM 2 (seq=False): (batch,      64)           — final hidden state only
    Dense 64         : (batch,      64)            — non-linear combination
    Output (softmax) : (batch,       4)            — 4 class probabilities
"""
def build_cnn_lstm(input_shape, n_classes,
                   cnn_filters, cnn_kernel_size, cnn_pool_size, cnn_dropout,
                   lstm_units, lstm_dropout, lstm_recurrent_dropout,
                   dense_units, learning_rate):
    """
    Build the CNN+LSTM hybrid model.

    Parameters
    ----------
    input_shape            : (T, C) — timesteps × features
    n_classes              : number of gesture classes
    cnn_filters            : list of int, filters per Conv1D block
                             e.g. [64, 128] → two blocks with 64 then 128 filters
    cnn_kernel_size        : int, kernel width for all Conv1D layers
                             e.g. 7 → each filter spans 7 consecutive timesteps
    cnn_pool_size          : int, MaxPooling1D pool size (2 → halves T after each block)
    cnn_dropout            : float, spatial dropout after each CNN block
    lstm_units             : list of int, units per LSTM layer
    lstm_dropout           : float, output dropout in LSTM
    lstm_recurrent_dropout : float, recurrent (Gal) dropout in LSTM
    dense_units            : list of int, units per Dense layer before softmax
    learning_rate          : float

    Returns
    -------
    model : compiled tf.keras.Model
    """
    # ── Input layer ───────────────────────────────────────────────────────────
    # Defines the expected shape of one sample (without the batch dimension).
    # shape = (TARGET_LEN, N_FEATURES)  e.g. (156, 180)
    inputs = keras.Input(shape=input_shape, name='sensor_input')
    x = inputs    # 'x' is a symbolic tensor; we thread it through each layer below

    # ── CNN stage ─────────────────────────────────────────────────────────────
    # Purpose: detect LOCAL temporal patterns and compress the time dimension.
    # Each iteration adds one Conv1D block: Conv → BatchNorm → ReLU → Pool → Dropout.
    #
    # Lecture 10 connection:
    #   Conv1D here is the exact same operation as the 2D convolution you learned,
    #   but the "image" has only ONE spatial dimension (time) instead of height×width.
    #   Each filter slides along the time axis across ALL input channels simultaneously
    #   (just like a 2D filter looks at all RGB channels at once).
    for i, n_filters in enumerate(cnn_filters):
        # Conv1D(filters, kernel_size, padding='same'):
        #   - filters     : how many different patterns to detect (e.g. 64)
        #   - kernel_size : width of the detection window in time (e.g. 7 timesteps)
        #   - padding='same': zero-pad the edges so output length = input length
        #                     (the MaxPool that follows will reduce length by 2)
        #   - use_bias=False: BatchNormalization below adds its own bias term,
        #                     so the Conv bias is redundant here
        x = layers.Conv1D(
            filters     = n_filters,     # number of learned filters (feature maps)
            kernel_size = cnn_kernel_size, # how many timesteps each filter covers
            padding     = 'same',        # preserve temporal length before pooling
            use_bias    = False,         # BN handles bias — no double-bias
            name        = f'conv1d_{i+1}'
        )(x)
        # After Conv1D: shape is (batch, T, n_filters)
        # 'T' is unchanged because padding='same'; n_filters channels replaces N_FEATURES

        # BatchNormalization: normalises activations across the batch and time dimensions.
        # This stabilises training, acts as mild regularisation, and allows higher LRs.
        # Internally: output = gamma * (x - mean)/std + beta
        # where gamma and beta are learned per-channel parameters.
        x = layers.BatchNormalization(name=f'bn_{i+1}')(x)

        # ReLU activation: max(0, x) — introduces non-linearity so the network can
        # represent more than just linear relationships between sensor channels.
        # (You studied ReLU in COMP3308 — same function, applied channel-wise here.)
        x = layers.Activation('relu', name=f'relu_cnn_{i+1}')(x)

        # MaxPooling1D: takes the maximum value in each non-overlapping window of
        # pool_size timesteps. This halves the temporal resolution (156→78→39 steps)
        # and provides translational invariance over small timing shifts.
        # Shape after pool: (batch, T//2, n_filters)
        x = layers.MaxPooling1D(pool_size=cnn_pool_size,
                                name=f'maxpool_{i+1}')(x)

        # Spatial Dropout: randomly zeroes entire filter channels during training
        # (rather than individual values). Prevents co-adaptation of feature maps.
        x = layers.Dropout(cnn_dropout, name=f'dropout_cnn_{i+1}')(x)

    # After CNN stage: shape is (batch, T', last_cnn_filter)
    # With default config: (batch, 39, 128)
    # This is already in the right shape for LSTM: (batch, timesteps, features)
    # The 39 "timesteps" each represent a compressed summary of ~4 raw timesteps.

    # ── LSTM stage ────────────────────────────────────────────────────────────
    # Purpose: model LONG-RANGE temporal dependencies in the CNN feature sequence.
    #
    # The LSTM processes the sequence one step at a time, left to right.
    # At each of the 39 compressed timesteps it sees:
    #   x_t   = the 128-dim CNN feature vector for that position
    #   h_t-1 = its own hidden state from the previous position
    # This allows it to "remember" what happened at step 5 when it reaches step 35.
    for i, units in enumerate(lstm_units):
        # return_sequences: True for all layers except the last.
        #   True  → output h_t at every step → shape (batch, 39, units)
        #           needed so the NEXT LSTM layer can process the full sequence
        #   False → output only the final h_T → shape (batch, units)
        #           collapses sequence into a single summary vector for the Dense head
        return_seq = (i < len(lstm_units) - 1)   # True for all but last LSTM

        x = layers.LSTM(
            units              = units,
            return_sequences   = return_seq,      # see explanation above
            dropout            = lstm_dropout,    # dropout on LSTM outputs (regularise)
            recurrent_dropout  = lstm_recurrent_dropout,  # Gal variational dropout
                                                          # applied to the recurrent
                                                          # state h_{t-1} at each step
            name               = f'lstm_{i+1}'
        )(x)
        # After LSTM 1 (return_seq=True) : shape (batch, 39, 128)
        # After LSTM 2 (return_seq=False): shape (batch, 64)

    # ── Dense head ────────────────────────────────────────────────────────────
    # A standard fully-connected classifier, identical to what you studied in
    # COMP3308. Takes the LSTM's 64-dim summary vector and predicts class scores.
    for i, units in enumerate(dense_units):
        x = layers.Dense(units, activation='relu',
                         name=f'dense_{i+1}')(x)   # (batch, 64) → (batch, 64)
        x = layers.Dropout(0.3, name=f'dropout_dense_{i+1}')(x)  # regularise

    # Final output layer: N_CLASSES units with softmax activation.
    # Softmax converts raw scores into a probability distribution over classes:
    #   output[i] ∈ (0,1) and sum(output) = 1
    # The predicted class is argmax(output).  (Identical to the softmax output
    # head you learned in COMP3308 multiclass classification.)
    outputs = layers.Dense(n_classes, activation='softmax',
                           name='output')(x)        # (batch, 4)

    # ── Assemble and compile ──────────────────────────────────────────────────
    model = keras.Model(inputs=inputs, outputs=outputs,
                        name='CNN_LSTM_Gesture')

    # Adam optimiser: adaptive learning rate algorithm that adjusts step sizes
    # per parameter. More effective than vanilla gradient descent for deep networks.
    # categorical_crossentropy: the standard loss for multiclass classification
    # with one-hot encoded targets (equivalent to -log(p_true_class) per sample).
    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = 'categorical_crossentropy',   # suitable for one-hot labels
        metrics   = ['accuracy'],                 # monitor classification accuracy
    )
    return model


# ── Instantiate the model with CONFIG hyperparameters ─────────────────────────
model = build_cnn_lstm(
    input_shape            = (TARGET_LEN, N_FEATURES),   # (156, 180) by default
    n_classes              = N_CLASSES,                  # 4 gestures
    cnn_filters            = CNN_FILTERS,                # [64, 128]
    cnn_kernel_size        = CNN_KERNEL_SIZE,            # 7
    cnn_pool_size          = CNN_POOL_SIZE,              # 2
    cnn_dropout            = CNN_DROPOUT,                # 0.2
    lstm_units             = LSTM_UNITS,                 # [128, 64]
    lstm_dropout           = LSTM_DROPOUT,               # 0.3
    lstm_recurrent_dropout = LSTM_RECURRENT_DROPOUT,     # 0.2
    dense_units            = DENSE_UNITS,                # [64]
    learning_rate          = LEARNING_RATE,              # 0.001
)

# Print the Keras model summary — shows each layer, its output shape, and param count.
# Study the 'Output Shape' column to verify the shape flow described above.
model.summary()

# ── Shape trace: visualise the CNN→LSTM temporal compression ──────────────────
# This explicitly computes how the time dimension shrinks through the CNN blocks.
print('\n── CNN → LSTM output shape trace ──')
T = TARGET_LEN
for filt in CNN_FILTERS:
    T = T // CNN_POOL_SIZE   # each MaxPool halves the time dimension
    print(f'  After Conv+Pool block ({filt} filters): ({T}, {filt})')
print(f'  → LSTM input: ({T}, {CNN_FILTERS[-1]})')
print(f'  (Original sequence: {TARGET_LEN} steps → compressed to {T} steps,'
      f' each summarising ~{TARGET_LEN // T} raw timesteps)')


---
## Cell 8 — Training

Three callbacks automate training hygiene:

| Callback | Monitors | Action |
|---|---|---|
| **EarlyStopping** | `val_accuracy` | Stops training if val accuracy hasn't improved for `EARLY_STOP_PATIENCE` epochs; restores the best weights found |
| **ReduceLROnPlateau** | `val_loss` | Halves the learning rate when val loss plateaus for 5 epochs, helping the optimiser escape shallow local minima |
| **ModelCheckpoint** | `val_accuracy` | Saves the model weights whenever a new best val accuracy is achieved, so the best checkpoint is always on disk |

**Why EarlyStopping prevents overfitting:**
Overfitting (your COMP3308 lectures) occurs when a model memorises the training set
and loses generalisation ability — training accuracy keeps rising but validation accuracy
stops improving or falls. EarlyStopping detects exactly this divergence and terminates
training, returning the weights from the epoch where val accuracy peaked.

After training, loss and accuracy curves are plotted. A healthy run shows:
- Both train and val loss decreasing together (no large gap = no overfitting)
- Both train and val accuracy converging to a high value


In [ ]:
"""
Configure training callbacks and run model.fit().

Inputs : model (compiled), X_train/y_train_oh, X_val/y_val_oh
Output : history object — contains per-epoch loss and accuracy for both sets
"""
# ── Define output path for the best checkpoint ────────────────────────────────
checkpoint_path = os.path.join(MODEL_DIR, f'{MODEL_NAME}_best.keras')

# ── Configure callbacks ───────────────────────────────────────────────────────
callbacks = [
    # EarlyStopping: monitors val_accuracy; if no improvement for PATIENCE epochs,
    # halt training and restore the weights from the best epoch automatically.
    EarlyStopping(
        monitor              = 'val_accuracy',   # metric to watch
        patience             = EARLY_STOP_PATIENCE,  # epochs to wait before stopping
        restore_best_weights = True,             # rollback to best epoch on stop
        verbose              = 1,               # print a message when triggered
    ),
    # ReduceLROnPlateau: if val_loss doesn't improve for 5 epochs, multiply LR by 0.5.
    # Helps the optimiser make finer adjustments when training slows down.
    ReduceLROnPlateau(
        monitor   = 'val_loss',   # watch validation loss
        factor    = 0.5,          # new_lr = lr * factor
        patience  = 5,            # epochs of no improvement before reducing
        min_lr    = 1e-6,         # lower bound — never go below this learning rate
        verbose   = 1,
    ),
    # ModelCheckpoint: saves model whenever a new best val_accuracy is achieved.
    # Even if training crashes later, the best weights are preserved on disk.
    ModelCheckpoint(
        filepath         = checkpoint_path,   # where to save the .keras file
        monitor          = 'val_accuracy',    # metric to track
        save_best_only   = True,             # only overwrite if this epoch is better
        verbose          = 0,
    ),
]

# ── Train the model ───────────────────────────────────────────────────────────
# model.fit() runs gradient descent for up to EPOCHS iterations over the training set.
# Each iteration:
#   1. Forward pass: compute predictions and loss on one mini-batch (BATCH_SIZE samples)
#   2. Backward pass: compute gradients of loss w.r.t. all weights (backpropagation)
#   3. Update: Adam adjusts each weight using its accumulated gradient statistics
# After each epoch, the model is evaluated on X_val / y_val_oh (no weight updates).
history = model.fit(
    X_train, y_train_oh,            # training data and one-hot labels
    validation_data = (X_val, y_val_oh),  # val set: tracked but not trained on
    epochs          = EPOCHS,       # maximum number of full passes through training set
    batch_size      = BATCH_SIZE,   # samples per gradient update
    callbacks       = callbacks,    # EarlyStopping, LR reduction, checkpoint
    verbose         = 1,            # print progress bar per epoch
)

print(f'\nBest model saved to: {checkpoint_path}')

# ── Training curves ───────────────────────────────────────────────────────────
# Plot loss and accuracy vs epoch to visually diagnose training behaviour.
# Ideal: both curves decrease/increase together without a large train-val gap.
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# ── Loss curve ────────────────────────────────────────────────────────────────
# Categorical cross-entropy loss: lower = better predictions.
# If train loss falls but val loss rises → overfitting.
ax1.plot(epochs_ran, hist['loss'],     color='#20808D', linewidth=1.8, label='Train')
ax1.plot(epochs_ran, hist['val_loss'], color='#A84B2F', linewidth=1.8,
         linestyle='--', label='Validation')
ax1.set_title('Categorical Cross-Entropy Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# ── Accuracy curve ────────────────────────────────────────────────────────────
# Proportion of correctly classified samples per epoch.
# Val accuracy is the metric we care about most — training accuracy can overfit.
ax2.plot(epochs_ran, hist['accuracy'],     color='#20808D', linewidth=1.8, label='Train')
ax2.plot(epochs_ran, hist['val_accuracy'], color='#A84B2F', linewidth=1.8,
         linestyle='--', label='Validation')
ax2.set_title('Classification Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.05)
ax2.legend()

plt.suptitle('CNN+LSTM Training History', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()

best_val_acc = max(hist['val_accuracy'])
print(f'Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')


---
## Cell 9 — Evaluation

Evaluate on the held-out **test set** — data the model has never seen during training
or hyperparameter selection. This gives the most honest estimate of real-world performance.

**Outputs:**
- **Test accuracy** — overall proportion of correct predictions across all 80 test samples.
- **Classification report** — per-class precision, recall, and F1 score. These are the
  metrics you studied in COMP3308:
  - *Precision*: of all samples predicted as class C, what fraction actually are C?
  - *Recall*: of all true class C samples, what fraction did the model find?
  - *F1*: harmonic mean of precision and recall — a single balanced score.
- **Confusion matrix** — an N×N grid where cell [i, j] counts test samples of true
  class i that were predicted as class j. The diagonal is correct predictions; any
  off-diagonal entries are misclassifications (and tell you *which* class was confused
  with which).


In [ ]:
"""
Evaluate the trained CNN+LSTM model on the held-out test set.

Produces:
  - Scalar test accuracy
  - Per-class classification report (precision, recall, F1)
  - Confusion matrix (raw counts and row-normalised / recall)
"""
# ── Get model predictions on the test set ─────────────────────────────────────
# model.predict() performs a forward pass only (no gradient computation).
# Output shape: (N_test, N_CLASSES) — a probability distribution over classes per sample.
y_pred_prob = model.predict(X_test, verbose=0)          # (80, 4) — class probabilities

# Take the class with the highest probability as the predicted label.
# argmax collapses the last dimension: (80, 4) → (80,) integer array.
y_pred      = np.argmax(y_pred_prob, axis=1)            # (80,) — predicted class indices
y_true      = y_test                                    # (80,) — true class indices

# ── Overall accuracy ──────────────────────────────────────────────────────────
# Proportion of test samples where predicted class == true class.
# From COMP3308: accuracy = (TP + TN) / total = correct / N_test
test_acc  = accuracy_score(y_true, y_pred)
print(f'Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

# ── Per-class classification report ──────────────────────────────────────────
# For each class this prints:
#   precision = TP / (TP + FP) — how many predicted positives are correct?
#   recall    = TP / (TP + FN) — how many true positives did we catch?
#   f1-score  = 2 * P * R / (P + R) — balanced measure of both
#   support   = number of true samples of this class in the test set
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# ── Confusion matrix heatmap ──────────────────────────────────────────────────
# cm[i, j] = number of samples of true class i predicted as class j.
# A perfect classifier would show all values on the diagonal (i == j).
cm      = confusion_matrix(y_true, y_pred)

# Row-normalise: divide each row by its sum so values represent recall per class.
# cm_norm[i, j] = fraction of true class i samples predicted as class j.
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(max(10, N_CLASSES * 1.0 * 2),
                                        max(7, N_CLASSES * 0.65)))

for ax, data, fmt, title in zip(
    axes,
    [cm,                          cm_norm],
    ['d',                          '.2f'],            # 'd' = integer, '.2f' = 2 decimals
    ['Counts',                     'Row-Normalised (Recall)'],
):
    sns.heatmap(
        data,
        annot      = True,           # show numeric value in each cell
        fmt        = fmt,            # formatting for annotation text
        cmap       = 'YlGnBu',       # colour scale: light = low, dark = high
        xticklabels= CLASS_NAMES,    # columns = predicted labels
        yticklabels= CLASS_NAMES,    # rows    = true labels
        linewidths = 0.4,            # thin grid lines between cells
        ax         = ax,
        cbar_kws   = {'shrink': 0.8},
    )
    ax.set_title(f'Confusion Matrix — {title}', fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.tick_params(axis='x', rotation=40, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle(f'CNN+LSTM  |  Test Acc: {test_acc*100:.2f}%',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()


---
## Cell 10 — Save Model and Results JSON

Saves two artefacts for downstream use:

1. **Full model** (`.keras` format) — contains the complete architecture, all trained
   weights, and the optimiser state. Load with `keras.models.load_model(path)` for
   inference or fine-tuning.

2. **`results.json`** — a structured dictionary of all key metrics and hyperparameters.
   This file is read by `00_Results_Comparison.ipynb` to produce the cross-model
   comparison table. Every notebook in this project writes the same JSON schema so
   the comparison notebook can load and compare them uniformly.


In [ ]:
"""
Persist the trained model and a structured results JSON.

Outputs:
  saved_models/03_cnn_lstm_final.keras  — full model (weights + architecture)
  results/03_cnn_lstm_results.json      — metrics + config for cross-model comparison
"""
# ── Save the final trained model ──────────────────────────────────────────────
# The .keras format is Keras's native format: it stores the architecture (as a JSON
# config), all weight tensors, and the optimiser state. This is portable across
# TensorFlow versions and can be loaded with: keras.models.load_model(path)
final_model_path = os.path.join(MODEL_DIR, f'{MODEL_NAME}_final.keras')
model.save(final_model_path)
print(f'Model saved : {final_model_path}')

# ── Build the results dictionary ──────────────────────────────────────────────
# This produces a serialisable dict with all info needed to compare models.
# classification_report(output_dict=True) returns the same content as the
# human-readable string version, but as a nested dict for programmatic access.
report_dict = classification_report(
    y_true, y_pred,
    target_names = CLASS_NAMES,
    output_dict  = True,    # return dict instead of formatted string
    digits       = 4,
)

results = {
    'model'              : 'CNN+LSTM',
    'notebook'           : '03_CNN_LSTM',
    'test_accuracy'      : float(test_acc),              # final held-out accuracy
    'best_val_accuracy'  : float(best_val_acc),          # best val accuracy during training
    'n_classes'          : N_CLASSES,
    'n_features'         : N_FEATURES,
    'target_len'         : TARGET_LEN,
    'n_train_samples'    : int(len(X_train)),
    'n_val_samples'      : int(len(X_val)),
    'n_test_samples'     : int(len(X_test)),
    'epochs_trained'     : len(history.history['loss']),  # actual epochs run (≤ EPOCHS)
    'class_names'        : CLASS_NAMES,
    'classification_report': report_dict,
    # Full hyperparameter snapshot — ensures the JSON is self-documenting
    'config': {
        'CNN_FILTERS'            : CNN_FILTERS,
        'CNN_KERNEL_SIZE'        : CNN_KERNEL_SIZE,
        'CNN_POOL_SIZE'          : CNN_POOL_SIZE,
        'CNN_DROPOUT'            : CNN_DROPOUT,
        'LSTM_UNITS'             : LSTM_UNITS,
        'LSTM_DROPOUT'           : LSTM_DROPOUT,
        'LSTM_RECURRENT_DROPOUT' : LSTM_RECURRENT_DROPOUT,
        'DENSE_UNITS'            : DENSE_UNITS,
        'EPOCHS'                 : EPOCHS,
        'BATCH_SIZE'             : BATCH_SIZE,
        'LEARNING_RATE'          : LEARNING_RATE,
        'FILTER_TYPE'            : FILTER_TYPE,
        'NORMALIZATION'          : NORMALIZATION,
        'PER_SAMPLE_NORM'        : PER_SAMPLE_NORM,
    },
}

# Serialise to JSON — json.dump handles Python dicts/lists/primitives.
# indent=2 produces human-readable formatting.
results_path = os.path.join(RESULTS_DIR, f'{MODEL_NAME}_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved: {results_path}')
print(f'\n── Summary ──────────────────────────────────────')
print(f'  Test accuracy       : {test_acc*100:.2f}%')
print(f'  Best val accuracy   : {best_val_acc*100:.2f}%')
print(f'  Epochs trained      : {len(history.history["loss"])}')
print(f'  Macro F1 (test)     : {report_dict["macro avg"]["f1-score"]*100:.2f}%')


---
## Cell 11 — CNN Activation Visualisation

### What Are We Visualising?

The CNN frontend acts as a *feature extractor* before the LSTM sees the data. By inspecting
the intermediate feature maps — the output of each `Conv1D` layer — we can understand
**what temporal patterns the CNN has learned to detect**.

For a single test sample:
- Each **filter** in a Conv1D layer produces one output channel (a 1D time series).
- **High activation values** indicate that the filter's learned pattern is strongly present
  at that point in time.
- The **first CNN layer** (conv1d_1) detects low-level patterns: sharp acceleration peaks,
  rapid flex angle changes, plateaus.
- The **second CNN layer** (conv1d_2) detects more abstract combinations of those patterns
  — e.g. "a peak followed by a sustained plateau" or "oscillation then stillness".

### Why the post-pooling shapes differ

After `conv1d_1 + maxpool_1`: temporal length halved (156 → 78 steps), 64 channels.
After `conv1d_2 + maxpool_2`: temporal length halved again (78 → 39 steps), 128 channels.
When you look at the feature map plots, the x-axis represents these compressed timesteps,
not the original 156. Each tick on the x-axis corresponds to ~130 ms of original motion.

### Interpreting the heatmaps

A bright horizontal band at a specific time range means many filters activate strongly
during that part of the gesture — the CNN considers that period most informative for
classification. The LSTM then models the *ordering* of these activations across the
39 compressed steps to make the final class decision.


In [ ]:
"""
Extract and visualise intermediate CNN feature maps for one test sample.

Strategy
--------
1. Build a secondary Keras model (activation_model) that shares weights with
   the trained model but outputs the activations of each CNN ReLU layer.
2. Feed one correctly-classified test sample through activation_model.
3. Plot each filter's output as a 1D time series (individual subplots).
4. Plot all filters together as a 2D heatmap (filters × compressed time).

This gives a visual answer to: "What did the CNN detect, and when?"
"""
# ── Build intermediate activation models ─────────────────────────────────────
# We want the output AFTER the ReLU (post-BN, post-activation) for each CNN block,
# because those are the actual feature maps that flow into the next layer.
# layer.name contains 'relu_cnn' for the activation layers we named explicitly.
relu_layers = [l for l in model.layers if 'relu_cnn' in l.name]
pool_layers = [l for l in model.layers if 'maxpool' in l.name]

# Extract output from each Conv1D layer and each MaxPool layer
cnn_layer_names = [l.name for l in model.layers
                   if isinstance(l, (layers.Conv1D,
                                     layers.Activation,
                                     layers.MaxPooling1D))]

# Build a sub-model that has the same input as the full model but outputs the
# activations of every ReLU and MaxPool layer simultaneously.
# This does NOT require re-training — it reuses the trained weights.
activation_model = keras.Model(
    inputs  = model.input,
    outputs = [l.output for l in relu_layers + pool_layers],
)

# ── Pick one test sample — prefer a correctly classified one ──────────────────
# Using a correctly classified sample ensures we're visualising a "successful"
# activation pattern rather than a failure case.
correct_mask = (y_pred == y_true)
sample_idx = np.argmax(correct_mask)            # index of first correct prediction
sample_x   = X_test[sample_idx:sample_idx+1]   # keep batch dimension: (1, T, F)
true_label = CLASS_NAMES[y_true[sample_idx]]
pred_label = CLASS_NAMES[y_pred[sample_idx]]

print(f'Visualising sample idx={sample_idx}  '
      f'true={true_label}  pred={pred_label}')

# Run the activation sub-model: returns a list of arrays, one per output layer.
activations = activation_model.predict(sample_x, verbose=0)

# ── Plot CNN feature maps: individual filter traces ───────────────────────────
# For each ReLU layer, plot each filter's output as a small 1D line plot.
# High activation at a time step means the filter's learned pattern fired there.
n_relu_layers = len(relu_layers)
N_FILTERS_SHOW = 16   # show first 16 filters per layer (keeps the figure manageable)

for layer_idx, (layer, act) in enumerate(zip(relu_layers, activations[:n_relu_layers])):
    act = act[0]   # remove batch dimension: (1, T', n_filters) → (T', n_filters)
    n_show = min(N_FILTERS_SHOW, act.shape[-1])
    T_prime = act.shape[0]   # compressed time steps (78 for layer 1, 39 for layer 2)

    ncols = 8
    nrows = int(np.ceil(n_show / ncols))

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(ncols * 1.8, nrows * 1.4),
                              sharex=True, sharey=False)
    axes = np.array(axes).reshape(-1)

    fig.suptitle(
        f'CNN Feature Maps — {layer.name}  '
        f'(shape: {T_prime}×{act.shape[-1]})'
        f'  [each x-tick ≈ {TARGET_LEN // T_prime} raw timesteps ≈ '
        f'{(TARGET_LEN / T_prime) / FS_HZ * 1000:.0f} ms]\n'
        f'Sample: "{true_label}"  |  First {n_show} of {act.shape[-1]} filters',
        fontsize=10, fontweight='bold', y=1.01
    )

    for fi in range(n_show):
        ax = axes[fi]
        fmap = act[:, fi]            # 1D activation trace for filter fi: shape (T',)
        ax.plot(fmap, color='#20808D', linewidth=0.9)
        # Fill under the curve to make high-activation regions visually prominent
        ax.fill_between(range(T_prime), fmap, alpha=0.25, color='#20808D')
        ax.set_title(f'F{fi}', fontsize=7, pad=2)
        ax.tick_params(labelsize=5)
        ax.set_xticks([])
        ax.set_yticks([])

    # Hide any unused subplot panels (if n_show is not a multiple of ncols)
    for fi in range(n_show, len(axes)):
        axes[fi].set_visible(False)

    plt.tight_layout()
    save_name = os.path.join(
        RESULTS_DIR,
        f'{MODEL_NAME}_cnn_features_{layer.name}.png'
    )
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_name}')

# ── Heatmap: all filters across time (global view) ────────────────────────────
# The heatmap shows ALL filters (rows) vs ALL compressed time steps (columns).
# Bright cells = strong activation; this reveals WHEN each filter fires.
# Filters that fire in similar time windows detect related features.
for layer_idx, (layer, act) in enumerate(zip(relu_layers, activations[:n_relu_layers])):
    act_map = act[0].T   # (n_filters, T') — transpose so filters are on the y-axis
    n_show  = min(64, act_map.shape[0])   # cap at 64 filters for readability

    fig, ax = plt.subplots(figsize=(12, max(3, n_show * 0.12)))
    im = ax.imshow(
        act_map[:n_show],
        aspect        = 'auto',      # stretch to fill the axes
        cmap          = 'viridis',   # perceptually uniform: dark=low, bright=high
        interpolation = 'nearest',
    )
    plt.colorbar(im, ax=ax, label='Activation magnitude')
    ax.set_title(
        f'Feature Map Heatmap — {layer.name}\n'
        f'Sample: "{true_label}"  |  First {n_show} filters × {act_map.shape[1]} timesteps',
        fontweight='bold'
    )
    ax.set_xlabel(f'Compressed time step (each ≈ {TARGET_LEN // act_map.shape[1]} raw steps)')
    ax.set_ylabel('Filter index')
    plt.tight_layout()
    save_name = os.path.join(
        RESULTS_DIR,
        f'{MODEL_NAME}_cnn_heatmap_{layer.name}.png'
    )
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_name}')

print('\nActivation visualisation complete.')
print('Interpretation guide:')
print('  Layer 1 (conv1d_1): detect low-level patterns — peaks, plateaus, transitions')
print('  Layer 2 (conv1d_2): detect combinations of those patterns across time')
print('  High-activation regions = time segments the CNN finds most discriminative')
print('  The LSTM then models the ordering of these compressed feature sequences,')
print('  deciding the final gesture class based on how the pattern sequence unfolds.')
